In [1]:
!pip install chromadb pypdf requests

In [2]:
import os
import requests
from pypdf import PdfReader
import chromadb

In [3]:
# -----------------------------
# 1. Download PDF from link
# -----------------------------
def download_pdf(pdf_url, output_path="sample.pdf"):
    response = requests.get(pdf_url)
    response.raise_for_status()

    with open(output_path, "wb") as file:
        file.write(response.content)

    return output_path


In [4]:


# -----------------------------
# 2. Extract text from PDF
# -----------------------------
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


In [5]:
# -----------------------------
# 3. Split text into chunks
# -----------------------------
def split_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap

    return chunks


In [7]:


# -----------------------------
# 4. Store chunks in ChromaDB
# -----------------------------
def store_in_chromadb(chunks):
    client = chromadb.PersistentClient(path="./pdf_chroma_db")

    collection = client.get_or_create_collection(
        name="pdf_knowledge_base"
    )

    ids = [f"chunk_{i}" for i in range(len(chunks))]

    collection.add(
        documents=chunks,
        ids=ids
    )

    return collection


# -----------------------------
# 5. Ask question from PDF
# -----------------------------
def ask_question(collection, question, n_results=3):
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    print("\nQuestion:")
    print(question)

    print("\nRelevant PDF Chunks:")
    for doc in results["documents"][0]:
        print("\n---")
        print(doc[:1000])

    print("\nSimple Answer:")
    print("Use the above retrieved PDF content as context for answering the question.")


# -----------------------------
# MAIN PROGRAM
# -----------------------------
if __name__ == "__main__":

    # OPTION 1: Use local PDF
    # Put your PDF in the same folder and give its name here
    pdf_path = "/content/sample_data/ML-Notes.pdf"

    # OPTION 2: Download PDF from link
    # Uncomment below lines if you want to use a PDF URL
    """
    pdf_url = "https://www.example.com/sample.pdf"
    pdf_path = download_pdf(pdf_url, "downloaded_document.pdf")
    """

    if not os.path.exists(pdf_path):
        print(f"PDF file not found: {pdf_path}")
        print("Please place your PDF in the same folder or use a valid PDF URL.")
        exit()

    print("Extracting text from PDF...")
    text = extract_text_from_pdf(pdf_path)

    print("Splitting text into chunks...")
    chunks = split_text(text)

    print("Storing chunks in ChromaDB...")
    collection = store_in_chromadb(chunks)

    print("PDF uploaded successfully into ChromaDB.")

    question = "What is the difference between Artificial Intelligence and Machine Learning?"
    ask_question(collection, question)

Extracting text from PDF...
Splitting text into chunks...
Storing chunks in ChromaDB...
PDF uploaded successfully into ChromaDB.

Question:
What is the difference between Artificial Intelligence and Machine Learning?

Relevant PDF Chunks:

---
 
  
DEPT. OF AIML , JNNCE 1 
 
 
MACHINE LEARNING STUDY MATERIAL, 
 
MODULE I: 
 
Definitions of Machine Learning 
 Machine Learning  is the science (and art) of programming computers so they 
can learn from data. 
 Machine Learning is the field of study that gives computers the ability to learn 
without being explicitly programmed. 
 A computer program is said to learn from experience E with respect to some task 
T and some performance measure P, if its performance on T, as measured by P, 
improves with experience E. 
 
Need to use Machine Learning 
 
Machine Learning is great for: 
 Problems for which existing solutions require a lot of hand -tuning or long lists of 
rules: one Machine Learning algorithm can often simplify code and perform bet